<a href="https://colab.research.google.com/github/Atul57/Data-Science-project/blob/main/Automatic_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
'/content/drive/MyDrive/cafe_sales_dirty.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/cafe_sales_dirty.csv'

In [ ]:
def automatic_data_cleaning(file_path):
    # Load data
    df = pd.read_csv(file_path)

    # Remove duplicates
    df = df.drop_duplicates()

    # Replace UNKNOWN and ERROR with NaN
    df = df.replace(["UNKNOWN", "ERROR"], np.nan)

    # Remove extra spaces from text columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

    # Convert numeric columns automatically, avoiding coercing entire string columns to NaN
    for col in df.columns:
        original_dtype = df[col].dtype

        # Attempt to convert to numeric
        converted_series = pd.to_numeric(df[col], errors='coerce')

        if original_dtype == 'object' and converted_series.isnull().all() and not df[col].isnull().all():
            # This column was likely a categorical string column, keep it as is (after previous cleaning)
            pass
        else:
            df[col] = converted_series

    # Fill missing values
    for col in df.columns:
        if df[col].dtype == 'object':
            if not df[col].dropna().empty: # Check if there are any non-NaN values to find a mode from
                df[col] = df[col].fillna(df[col].mode()[0])
            # else: if all values are NaN, no mode can be found, they remain NaN
        else:
            df[col] = df[col].fillna(df[col].median()) # Use median for numerical columns

    return df

# Function Call
cleaned_df = automatic_data_cleaning("/content/drive/MyDrive/cafe_sales_dirty.csv")


cleaned_df.to_csv(
    '/content/drive/MyDrive/cafe_sales_cleaned_ADC.csv',
    index=False
)

print("Cleaned file saved successfully!")
print(cleaned_df.head())
print(cleaned_df.isnull().sum())

Cleaned file saved successfully!
  Transaction ID    Item  Quantity  Price Per Unit  Total Spent  \
0    TXN_1961373  Coffee       2.0             2.0          4.0   
1    TXN_4977031    Cake       4.0             3.0         12.0   
2    TXN_4271903  Cookie       4.0             1.0          8.0   
3    TXN_7034554   Salad       2.0             5.0         10.0   
4    TXN_3160411  Coffee       2.0             2.0          4.0   

   Payment Method  Location Transaction Date  
0     Credit Card  Takeaway       2023-09-08  
1            Cash  In-store       2023-05-16  
2     Credit Card  In-store       2023-07-19  
3  Digital Wallet  Takeaway       2023-04-27  
4  Digital Wallet  In-store       2023-06-11  
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64
